In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pyspark.sql.functions as F
from common_utils.generic_functions import rename_columns , casting , drop_columns
from common_utils.scd_functions import scd_type1
from common_utils.logging import get_logger
import json
from pyspark.sql.window import Window

In [0]:
# ============================================================
# 0. Setting up logger
# ============================================================

logger = get_logger("b2s_sales.ipynb")
logger.info("Session started")


# ============================================================
# 1. Config Path
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

print(config_path)
logger.info("Setting up configurations")


# ============================================================
# 2. Reading Config
# ============================================================

with open(config_path, "r") as file:
    config = json.load(file)


# ============================================================
# 3. Config Selection
# ============================================================

source_config = config["source"]

source_name = source_config["source_name"]

target_table = config["target"]["target_table"]


# ============================================================
# 4. Sales Transformations
# ============================================================

def sales_transformations(df, config):

    # --------------------------------------------------------
    # Product JSON schema
    # --------------------------------------------------------

    schema = StructType([
        StructField("curr", StringType(), True),
        StructField("id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("price", IntegerType(), True),
        StructField("qty", IntegerType(), True),
        StructField("unit", StringType(), True)
    ])


    # --------------------------------------------------------
    # Parse and flatten product JSON
    # --------------------------------------------------------

    df = (
        df.withColumn(
            "product",
            F.from_json(F.col("product"), schema)
        )
        .select(
            "customer_id",
            "customer_name",
            "product_name",
            "order_date",
            "product_category",
            "product.curr",
            "product.id",
            "product.name",
            "product.price",
            "product.qty",
            "product.unit",
            "total_price",
            "last_update_timestamp",
            "file_path"
        )
    )


    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    df = rename_columns(
        df,
        config.get("rename_columns", {})
    )


    # --------------------------------------------------------
    # Cast columns
    # --------------------------------------------------------

    df = casting(
        df,
        config.get("casting", {})
    )


    # --------------------------------------------------------
    # Drop columns
    # --------------------------------------------------------

    drop = config.get("drop_columns", [])

    df = drop_columns(df, drop)


    # --------------------------------------------------------
    # Deduplication
    # --------------------------------------------------------

    dedup = config.get("deduplication", {})

    if dedup.get("enabled", False):

        logger.info(
            "[%s] Removing duplicate records",
            source_name
        )

        order_expressions = []

        for item in dedup["order_by"]:

            col = F.col(item["column"])

            direction = item.get(
                "direction",
                "asc"
            ).lower()

            nulls = item.get(
                "nulls",
                "last"
            ).lower()


            if direction == "desc":

                col = (
                    col.desc_nulls_last()
                    if nulls == "last"
                    else col.desc_nulls_first()
                )

            else:

                col = (
                    col.asc_nulls_last()
                    if nulls == "last"
                    else col.asc_nulls_first()
                )


            order_expressions.append(col)


        window = (
            Window
            .partitionBy(*dedup["key_columns"])
            .orderBy(*order_expressions)
        )


        df = (
            df
            .withColumn(
                "rn",
                F.row_number().over(window)
            )
            .filter(F.col("rn") == 1)
            .drop("rn")
        )


    # --------------------------------------------------------
    # Select final columns
    # --------------------------------------------------------

    logger.info(
        "[%s] Selecting final silver columns",
        source_name
    )

    df = df.select(
        *config.get("select_columns", [])
    )


    # --------------------------------------------------------
    # Return transformed dataframe
    # --------------------------------------------------------

    return df


# ============================================================
# 5. Reading Bronze Table
# ============================================================

df = spark.read.table(
    source_config["source_table"]
)


# ============================================================
# 6. Run Transformations
# ============================================================

cleaned_sales = sales_transformations(
    df,
    config
)


logger.info(
    "[%s] Sales transformation completed",
    source_name
)


# ============================================================
# 7. SCD Type 1 / Initial Load
# ============================================================

scd_type = 1

key_columns = [
    "customer_id",
    "product_id",
    "order_date"
]


if spark.catalog.tableExists(target_table):

    if scd_type == 1:

        logger.info(
            "[%s] Starting SCD Type 1",
            source_name
        )

        scd_type1(
            spark,
            cleaned_sales,
            target_table,
            key_columns
        )

    else:

        raise ValueError(
            f"Unsupported SCD type [{scd_type}] "
            f"for source [{source_config['source_name']}]"
        )


else:

    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS retail_data_platform.silver"
    )

    (
        cleaned_sales
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )